# Lesson 05 — Vector Search

We have frame embeddings from Lesson 04. Now we search them **by text**.

The idea:
1. Encode a text query with CLIP → get a 512-d vector
2. Compute cosine similarity between the query vector and every frame vector
3. Return the top-k most similar frames

No Batch job this lesson — search is fast enough to run locally on CPU.

> **No local GPU?** That's fine. CLIP's text encoder is small and runs on CPU in <1 second.

## Step 1 — Load environment & download embeddings

In [ ]:
import io, json, os
import boto3, numpy as np
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
S3_BUCKET = os.environ["S3_BUCKET"]

s3 = boto3.client("s3")

obj        = s3.get_object(Bucket=S3_BUCKET, Key="embeddings/sample/embeddings.npy")
embeddings = np.load(io.BytesIO(obj["Body"].read()))   # (N, 512)

obj        = s3.get_object(Bucket=S3_BUCKET, Key="embeddings/sample/frame_keys.json")
frame_keys = json.loads(obj["Body"].read())

print(f"Loaded {embeddings.shape[0]} frame embeddings, each {embeddings.shape[1]}-dimensional")

## Step 2 — Load the CLIP text encoder

In [ ]:
import clip, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model, _ = clip.load("ViT-B/32", device=device)
model.eval()
print("CLIP loaded.")

## Step 3 — Search helper

This function encodes a text query with CLIP and returns the top-k frame indices.

In [ ]:
def search(query: str, top_k: int = 3) -> list[int]:
    """Return the top_k frame indices most similar to the text query."""
    # Encode the query text
    tokens       = clip.tokenize([query]).to(device)
    with torch.no_grad():
        text_vec = model.encode_text(tokens)           # shape: (1, 512)
        text_vec = text_vec / text_vec.norm(dim=-1, keepdim=True)   # normalise

    text_vec_np  = text_vec.cpu().numpy().flatten()    # (512,)

    # Cosine similarity: dot product (embeddings are already unit-normalised)
    scores = embeddings @ text_vec_np                  # (N,)

    # Return indices of top_k highest scores
    top_indices = scores.argsort()[::-1][:top_k]
    return top_indices.tolist(), scores

print("Search function ready.")

## Step 4 — Search! (edit the query and re-run)

Try: `"outdoor scene"`, `"close-up face"`, `"text on screen"`, `"dark scene"`, `"a busy street"`

In [ ]:
QUERY = "outdoor scene with trees"   # ← change me!

top_indices, scores = search(QUERY, top_k=3)

print(f"Query: '{QUERY}'")
for rank, idx in enumerate(top_indices):
    print(f"  #{rank+1}  frame {idx:05d}  score={scores[idx]:.3f}  key={frame_keys[idx]}")

## Step 5 — Show the top-3 matching frames

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, idx in zip(axes, top_indices):
    key = frame_keys[idx]
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    img = Image.open(io.BytesIO(obj["Body"].read()))
    ax.imshow(img)
    ax.set_title(f"Frame {idx}\nScore: {scores[idx]:.3f}", fontsize=9)
    ax.axis("off")

plt.suptitle(f"Top-3 results for: '{QUERY}'", fontsize=12)
plt.tight_layout()
plt.show()

## Key Takeaway

> We searched through all frames using **plain English**, with no labels, no metadata, no keyword index.  
> The only thing that made this work is the embedding created by CLIP on the GPU.

This pattern — embed → store → search by cosine similarity — powers semantic search, RAG, and image search at every major tech company.

---

## Next lesson → [06 — Full Pipeline](../06-full-pipeline/notebook.ipynb)

We'll string lessons 03 and 04 together with Batch job dependencies — one command runs the full pipeline.